# El precio de la autonomía
## Análisis del precio, la marca y la calidad de las motos, bicicletas y patinetes eléctricos anunciados en La Habana

**Autora:** Adriana Amador Verdecia · Ciencia de Datos, primer año · Universidad de La Habana · Facultad de Matemática y Computación (MATCOM)

**Lucy tiene veinte años y estudia en la Facultad de Matemática y Computación de la Universidad de La Habana.** Cada mañana debe recorrer una distancia considerable para llegar a la facultad, y el costo económico y el tiempo que exige el transporte público se han convertido en una carga difícil de sostener. Ante esa situación, ha comenzado a considerar la compra de un vehículo eléctrico, con el propósito de recorrer esos trayectos de forma más económica y predecible. Ha reunido para ello un presupuesto de **1.500 dólares**, con el cual le sería posible acceder a cualquiera de las categorías que hoy se ofrecen en el mercado: motos, bicicletas o patinetes eléctricos.

**El obstáculo, sin embargo, no es económico sino informativo:** los anuncios publican casi solo el precio, y la autonomía o el tipo de batería aparecen con mucha menos frecuencia y sin un criterio común que permita compararlos. Frente a un catálogo de cientos de publicaciones, no hay forma de saber cuánto rinde lo que se compra. **El objetivo general de este proyecto es determinar qué variables explican el precio de los vehículos eléctricos privados en Cuba y hasta qué punto es posible medir su calidad con la información que el mercado publica**; para lograrlo, se caracterizará la oferta y los precios de cada categoría, se medirá su autonomía y su relación con el precio, se evaluará qué se consigue con un presupuesto determinado y se repetirá el ejercicio en cada fuente, hasta recomendar qué vehículo elegir y dónde comprar. *El estudio se apoya en fuentes distintas: por ahora el JSON de Revolico; más adelante, los demás sitios de venta informal, las encuestas y los mercados donde se venden estos vehículos.*

### Pregunta de investigación

> ¿Qué variables explican el precio de los vehículos eléctricos privados en Cuba y hasta qué punto es posible medir su calidad con la información que publica el mercado?

In [ ]:
import json
import statistics as stat

import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110

RUTA = "data/anuncios_revolico.json"
ORDEN = ["Moto electrica", "Bicicleta electrica", "Patinete electrico"]
ETIQUETAS = {
    "Moto electrica": "Moto eléctrica",
    "Bicicleta electrica": "Bicicleta eléctrica",
    "Patinete electrico": "Patinete eléctrico",
}
COLORES = {
    "Moto electrica": "#C1443B",
    "Bicicleta electrica": "#2E6F95",
    "Patinete electrico": "#3E8E5A",
}

with open(RUTA, encoding="utf-8") as f:
    anuncios = json.load(f)["anuncios"]

print(f"Anuncios cargados: {len(anuncios)}")
print(f"Categorías: {[ETIQUETAS[c] for c in ORDEN]}")

def grupo(categoria):
    """Devuelve los anuncios de una categoría."""
    return [a for a in anuncios if a["tipo_vehiculo"] == categoria]

def tabla(filas, titulo):
    """Imprime una tabla de filas (listas) alineada por columnas."""
    encabezados, cuerpo = filas[0], filas[1:]
    anchos = [max(len(str(fila[i])) for fila in filas) for i in range(len(encabezados))]
    separador = "-+-".join("-" * a for a in anchos)
    print(f"\n{titulo}")
    print(" | ".join(str(e).ljust(a) for e, a in zip(encabezados, anchos)))
    print(separador)
    for fila in cuerpo:
        print(" | ".join(str(v).ljust(a) for v, a in zip(fila, anchos)))

## Análisis 1 — Precio promedio por categoría

*Este gráfico compara el precio promedio de las tres categorías en dólares. La barra clara junto a cada una marca la mediana, que casi coincide con el promedio: eso indica que ningún precio extremo está deformando el cálculo.*

In [ ]:
filas = [["Categoría", "Anuncios", "Promedio", "Mediana", "Mínimo", "Máximo"]]

for categoria in ORDEN:
    precios = [a["precio"] for a in grupo(categoria)]
    filas.append([
        ETIQUETAS[categoria], len(precios),
        f"{stat.mean(precios):,.0f}".replace(",", "."),
        f"{stat.median(precios):,.0f}".replace(",", "."),
        f"{min(precios):,.0f}".replace(",", "."),
        f"{max(precios):,.0f}".replace(",", "."),
    ])

tabla(filas, "Precios por categoría (USD)")

# Valores atípicos por el método del rango intercuartílico.
# No se eliminan del dataset: se reportan para mostrar que no deforman el promedio.
filas_at = [["Categoría", "Atípicos", "Promedio", "Promedio sin atípicos", "Cambio"]]
cambios = []

for categoria in ORDEN:
    precios = [a["precio"] for a in grupo(categoria)]
    q1, q3 = stat.quantiles(precios, n=4)[0], stat.quantiles(precios, n=4)[2]
    limite_iqr = 1.5 * (q3 - q1)
    atipicos = [p for p in precios if p < q1 - limite_iqr or p > q3 + limite_iqr]
    sin_atipicos = [p for p in precios if not p < q1 - limite_iqr and not p > q3 + limite_iqr]
    cambios.append(stat.mean(sin_atipicos) - stat.mean(precios))
    filas_at.append([
        ETIQUETAS[categoria], len(atipicos),
        f"{stat.mean(precios):,.0f}".replace(",", "."),
        f"{stat.mean(sin_atipicos):,.0f}".replace(",", "."),
        f"{cambios[-1]:+,.0f}".replace(",", "."),
    ])

tabla(filas_at, "Atípicos de precio (rango intercuartílico)")

nombres = [ETIQUETAS[c] for c in ORDEN]
promedios = [stat.mean([a["precio"] for a in grupo(c)]) for c in ORDEN]
medianas = [stat.median([a["precio"] for a in grupo(c)]) for c in ORDEN]
conteos = [len(grupo(c)) for c in ORDEN]

fig, ax = plt.subplots(figsize=(8, 5))
posiciones = list(range(len(nombres)))
ancho = 0.38

ax.bar([p - ancho / 2 for p in posiciones], promedios, ancho,
       label="Promedio", color="#2E6F95", edgecolor="white")
ax.bar([p + ancho / 2 for p in posiciones], medianas, ancho,
       label="Mediana", color="#9DC3D4", edgecolor="white")

for p, valor in zip([q - ancho / 2 for q in posiciones], promedios):
    ax.text(p, valor + 60, f"{valor:,.0f}".replace(",", "."),
            ha="center", fontsize=9, fontweight="bold")
for p, valor in zip([q + ancho / 2 for q in posiciones], medianas):
    ax.text(p, valor + 60, f"{valor:,.0f}".replace(",", "."),
            ha="center", fontsize=9, color="#4A4A4A")

ax.set_xticks(posiciones)
ax.set_xticklabels([f"{n}\n(n = {c} anuncios)" for n, c in zip(nombres, conteos)])
ax.set_ylabel("Precio (USD)")
ax.set_ylim(0, 3000)
ax.set_title("Precio promedio y mediana por categoría\n224 anuncios de La Habana publicados en Revolico")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Análisis 2 — Qué puede comprar Lucy con 1.500 dólares

*Cada punto es un anuncio, con el precio en el eje horizontal y la autonomía en el vertical. La franja sombreada marca los 1.500 dólares de Lucy: los puntos dentro son los que puede comprar, los de la derecha quedan fuera de su alcance.*

In [ ]:
PRESUPUESTO = 1500

filas = [["Categoría", "Anuncios", "Alcanzables", "Precio mín.", "Precio mediana",
          "Aut. mín.", "Aut. mediana", "Aut. máx."]]

for categoria in ORDEN:
    todos = grupo(categoria)
    alcanzables = [a for a in todos if a["precio"] <= PRESUPUESTO]
    if alcanzables:
        precios = [a["precio"] for a in alcanzables]
        autonomias = [a["autonomia_km"] for a in alcanzables]
        filas.append([
            ETIQUETAS[categoria], len(todos), len(alcanzables),
            f"{min(precios):,.0f}".replace(",", "."),
            f"{stat.median(precios):,.0f}".replace(",", "."),
            min(autonomias),
            f"{stat.median(autonomias):.0f}",
            max(autonomias),
        ])
    else:
        filas.append([ETIQUETAS[categoria], len(todos), 0, "-", "-", "-", "-", "-"])

tabla(filas, f"Anuncios dentro del presupuesto de {PRESUPUESTO} USD")

# Las motos baratas que sí le alcanzan a Lucy: el caso que se ve en el gráfico
print("\nMotos dentro del presupuesto, detalle:")
for a in sorted(grupo("Moto electrica"), key=lambda x: x["precio"]):
    if a["precio"] <= PRESUPUESTO:
        print(f"  {a['precio']:>7,.0f} USD | {a['autonomia_km']:>3d} km | {a['marca']}")

LIMITE_X = (0, 3600)
LIMITE_Y = (0, 175)

fig, ejes = plt.subplots(1, 3, figsize=(13.5, 5), sharex=True, sharey=True)

for eje, categoria in zip(ejes, ORDEN):
    todos = grupo(categoria)
    alcanzables = [a for a in todos if a["precio"] <= PRESUPUESTO]

    eje.axvspan(*LIMITE_X, 0, PRESUPUESTO, color="#F2F2F2", zorder=0)
    eje.scatter([a["precio"] for a in todos], [a["autonomia_km"] for a in todos],
                s=42, facecolor="white", edgecolor="#BFBFBF", linewidth=1.1, zorder=2)
    eje.scatter([a["precio"] for a in alcanzables], [a["autonomia_km"] for a in alcanzables],
                s=52, color=COLORES[categoria], edgecolor="white", linewidth=0.8, zorder=3)

    eje.axvline(PRESUPUESTO, color="#C1443B", linestyle="--", linewidth=1.4, zorder=1)
    eje.set_xlim(*LIMITE_X)
    eje.set_ylim(*LIMITE_Y)
    eje.set_title(f"{ETIQUETAS[categoria]}\n{len(alcanzables)} de {len(todos)} anuncios dentro del presupuesto",
                  fontsize=10)
    eje.set_xlabel("Precio (USD)")
    eje.set_xticks([0, 1000, 2000, 3000])
    eje.spines[["top", "right"]].set_visible(False)

ejes[0].set_ylabel("Autonomía (km)")
ejes[0].text(PRESUPUESTO * 0.96, LIMITE_Y[1] * 0.95, "1.500 USD", ha="right",
             fontsize=9, color="#C1443B", fontweight="bold")
fig.suptitle("Precio y autonomía frente al presupuesto de Lucy (1.500 USD)")
plt.tight_layout()
plt.show()

## Conclusiones

*(Espacio para escribir la interpretación de los resultados y las recomendaciones.)*